In [1]:
import pandas as pd

# =========================================
# 0) 경로만 수정
# =========================================
in_path  = r"C:\ai\travel_dataset\전국관광지정보표준데이터.csv"          # 예: r"C:\ai\...\전국관광지정보표준데이터.csv"
out_path = r"C:\ai\travel_dataset\전국관광지정보표준데이터_부산서울제주.csv"

TARGETS = ["부산", "서울", "제주"]  # 포함 키워드

# =========================================
# 1) CSV 로드(인코딩 자동)
# =========================================
def read_csv_auto_encoding(path):
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    # 마지막 fallback (깨진문자 무시)
    return pd.read_csv(path, encoding="utf-8", errors="ignore")

df = read_csv_auto_encoding(in_path)

# =========================================
# 2) '소재지' 관련 컬럼 자동 탐색
#    - 보통: 소재지도로명주소, 소재지지번주소, 소재지 등
# =========================================
cand_cols = [c for c in df.columns if "소재지" in c]  # '소재지'가 들어간 컬럼들
if not cand_cols:
    # 혹시 컬럼명이 다르면 여기서 넓게 잡기(원하면 수정)
    cand_cols = [c for c in df.columns if ("주소" in c) or ("소재" in c)]

if not cand_cols:
    raise ValueError("소재지/주소 관련 컬럼을 찾지 못했어요. df.columns 출력해서 컬럼명을 확인해 주세요.")

# =========================================
# 3) 필터: 후보 컬럼들 중 하나라도 TARGETS 포함하면 남김
# =========================================
pattern = "|".join(TARGETS)  # '부산|서울|제주'

mask = False
for c in cand_cols:
    mask = mask | df[c].astype(str).str.contains(pattern, na=False)

df_filtered = df[mask].copy()

# =========================================
# 4) 저장(엑셀 호환 위해 utf-8-sig 추천)
# =========================================
df_filtered.to_csv(out_path, index=False, encoding="utf-8-sig")

print("원본 행 수:", len(df))
print("필터 후 행 수:", len(df_filtered))
print("사용한 주소 컬럼:", cand_cols)
print("저장 완료:", out_path)


원본 행 수: 854
필터 후 행 수: 155
사용한 주소 컬럼: ['소재지도로명주소', '소재지지번주소']
저장 완료: C:\ai\travel_dataset\전국관광지정보표준데이터_부산서울제주.csv


In [6]:
import pandas as pd

# =========================================
# 0) 경로만 수정
# =========================================
in_path  = r"C:\ai\travel_dataset\전국관광지정보표준데이터_부산서울제주.csv"  # 예: r"C:\ai\...\전국관광지정보표준데이터_부산서울제주.csv"
out_path = r"C:\ai\travel_dataset\전국관광지정보표준데이터_부산서울제주_최종주소_도시추가.csv"

# =========================================
# 1) CSV 로드(인코딩 자동)
# =========================================
def read_csv_auto_encoding(path):
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path, encoding="utf-8", errors="ignore")

df = read_csv_auto_encoding(in_path)

# =========================================
# 2) 컬럼명 설정 (표준데이터 기본 컬럼명)
# =========================================
road_col  = "소재지도로명주소"
jibun_col = "소재지지번주소"

if road_col not in df.columns or jibun_col not in df.columns:
    raise ValueError(f"필요 컬럼이 없어요.\n현재 컬럼: {list(df.columns)}")

# =========================================
# 3) 도로명주소 빈칸/NaN이면 지번주소로 채움 + 최종주소 생성
# =========================================
road  = df[road_col].astype(str).str.strip()
jibun = df[jibun_col].astype(str).str.strip()

road_is_empty  = df[road_col].isna()  | road.eq("")  | road.str.lower().eq("nan")
jibun_is_empty = df[jibun_col].isna() | jibun.eq("") | jibun.str.lower().eq("nan")

# ✅ 최종 소재지주소: 도로명이 있으면 도로명, 없으면 지번
df["소재지주소_최종"] = df[road_col]
df.loc[road_is_empty, "소재지주소_최종"] = df.loc[road_is_empty, jibun_col]
df.loc[road_is_empty & jibun_is_empty, "소재지주소_최종"] = pd.NA

# (옵션) 도로명주소 컬럼 자체를 채워넣고 싶으면 아래 주석 해제
# df.loc[road_is_empty, road_col] = df.loc[road_is_empty, jibun_col]

# =========================================
# 4) 최종주소로 도시 파싱(서울/부산/제주)
#    - 이미 이 파일은 3개만 남겨둔 상태라 간단히 포함 여부로 판정
# =========================================
targets = ["서울", "부산", "제주"]

def parse_city(addr):
    if pd.isna(addr):
        return pd.NA
    s = str(addr)
    for t in targets:
        if t in s:
            return t
    return pd.NA

df["도시"] = df["소재지주소_최종"].apply(parse_city)

# 나라 컬럼도 같이 만들고 싶으면 (필요시)
df["나라"] = "대한민국"

# =========================================
# 5) 저장
# =========================================
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("저장 완료:", out_path)
print("도로명주소 빈칸(대체 시도 대상) 건수:", int(road_is_empty.sum()))
print("도로명+지번 둘다 빈 행 수:", int((road_is_empty & jibun_is_empty).sum()))
print("도시별 건수:\n", df["도시"].value_counts(dropna=False))


저장 완료: C:\ai\travel_dataset\전국관광지정보표준데이터_부산서울제주_최종주소_도시추가.csv
도로명주소 빈칸(대체 시도 대상) 건수: 12
도로명+지번 둘다 빈 행 수: 0
도시별 건수:
 도시
부산    78
서울    64
제주    13
Name: count, dtype: int64


In [8]:
import pandas as pd
import re

# =========================================
# 0) 경로만 수정
# =========================================
in_path  = r"C:\ai\travel_dataset\전국관광지정보표준데이터_부산서울제주_최종주소_도시추가.csv"
out_path = r"C:\ai\travel_dataset\전국관광지정보표준데이터_부산서울제주_도시나라위경도_이용자수.csv"

# =========================================
# 1) CSV 로드(인코딩 자동)
# =========================================
def read_csv_auto_encoding(path):
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path, encoding="utf-8", errors="ignore")

df = read_csv_auto_encoding(in_path)

# =========================================
# 2) 컬럼 자동 탐색 유틸
# =========================================
def find_col(cols, keywords):
    for c in cols:
        cu = str(c).upper()
        for k in keywords:
            if k.upper() in cu:
                return c
    return None

# 위도/경도
lat_col = find_col(df.columns, ["위도", "LAT", "LATITUDE"])
lon_col = find_col(df.columns, ["경도", "LON", "LONGITUDE"])
if lat_col is None or lon_col is None:
    raise ValueError(f"위도/경도 컬럼을 못 찾았어요. 현재 컬럼: {list(df.columns)}")

# 도시/나라
for c in ["도시", "나라"]:
    if c not in df.columns:
        raise ValueError(f"'{c}' 컬럼이 없어요. (도시/나라 생성된 파일을 입력으로 넣어야 해요)\n현재 컬럼: {list(df.columns)}")

# =========================================
# 3) 수용인원(=이용자수로 바꿀 대상) 컬럼 찾기
#    - 파일마다 이름이 다를 수 있어 키워드로 탐색
# =========================================
cap_col = find_col(df.columns, [
    "수용", "수용인원", "수용인원수", "수용가능", "수용가능인원",
    "정원", "CAPACITY", "CAPA"
])

# 못 찾으면 후보 컬럼을 보여주고, 이용자수는 NA로 만들기
if cap_col is None:
    print("⚠️ 수용인원 관련 컬럼을 자동으로 못 찾았어요.")
    print("   컬럼 목록에서 '수용/정원' 관련이 있는지 확인하세요:")
    cand = [c for c in df.columns if any(k in str(c) for k in ["수용", "정원", "인원", "capacity", "CAPACITY"])]
    print("   후보:", cand)
    df["이용자수"] = pd.NA
else:
    # =========================================
    # 4) 수용인원 -> 이용자수 (숫자 정리)
    #    - 쉼표/공백/단위/문자 섞인 경우 숫자만 뽑아 변환
    # =========================================
    def to_number(x):
        if pd.isna(x):
            return pd.NA
        s = str(x).strip()
        if s == "" or s.lower() == "nan":
            return pd.NA
        # 예: "1,200명", "약 300", "300~500" -> 첫 숫자만 추출
        m = re.findall(r"\d+", s.replace(",", ""))
        if not m:
            return pd.NA
        return int(m[0])

    df["이용자수"] = df[cap_col].apply(to_number)

# =========================================
# 5) 최종 컬럼만 남기기
# =========================================
out_df = df[["도시", "나라", lat_col, lon_col, "이용자수"]].copy()
out_df = out_df.rename(columns={lat_col: "위도", lon_col: "경도"})

# 위도/경도 숫자 변환
out_df["위도"] = pd.to_numeric(out_df["위도"], errors="coerce")
out_df["경도"] = pd.to_numeric(out_df["경도"], errors="coerce")

# =========================================
# 6) 저장
# =========================================
out_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("저장 완료:", out_path)
print("남긴 컬럼:", list(out_df.columns))
print("행 수:", len(out_df))
print("이용자수 결측:", int(out_df["이용자수"].isna().sum()))


저장 완료: C:\ai\travel_dataset\전국관광지정보표준데이터_부산서울제주_도시나라위경도_이용자수.csv
남긴 컬럼: ['도시', '나라', '위도', '경도', '이용자수']
행 수: 155
이용자수 결측: 0


In [19]:
import pandas as pd
import numpy as np
import requests
import time
import math

# =========================================================
# ✅ 0) 여기만 수정
# =========================================================
in_path  = r"C:\ai\travel_dataset\전국관광지정보표준데이터_부산서울제주_도시나라위경도_이용자수.csv"
out_path = r"C:\ai\travel_dataset\전국관광지정보표준데이터_부산서울제주_도시나라위경도_이용자수_해변10km.csv"

거리기준_m = 10_000
SLEEP_SEC = 4.0        # Overpass 과호출 방지 (3~6 추천)
EXTRA_KM  = 5          # bbox 여유 km
CAND_KM   = 35         # 거리 계산 후보 필터용(속도)

# Overpass 엔드포인트(하나 막히면 다음으로)
OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

# 해변 후보 태그(한국에서 태그가 다양한 경우 대비)
# ※ 서울은 어차피 0으로 강제하므로 부산/제주만 의미 있음
BEACH_TAGS = [
    ("natural", "beach"),
    ("landuse", "beach"),
    ("tourism", "beach"),
    ("natural", "sand"),
    ("leisure", "beach_resort"),
]

# =========================================================
# ✅ 1) CSV 로드(구버전 pandas 호환)
# =========================================================
def read_csv_auto_encoding(path):
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    # 마지막 fallback: open에서 errors='ignore' 처리
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return pd.read_csv(f)

df = read_csv_auto_encoding(in_path)

need_cols = ["도시", "나라", "위도", "경도"]
for c in need_cols:
    if c not in df.columns:
        raise ValueError(f"'{c}' 컬럼이 없습니다. 현재 컬럼: {list(df.columns)}")

df["위도"] = pd.to_numeric(df["위도"], errors="coerce")
df["경도"] = pd.to_numeric(df["경도"], errors="coerce")

# =========================================================
# ✅ 2) km -> 위경도 버퍼(대략)
# =========================================================
def km_to_deg_buffer(lat_mean, km):
    lat_buf = km / 111.0
    lon_buf = km / (111.0 * max(0.2, math.cos(math.radians(lat_mean))))
    return lat_buf, lon_buf

# =========================================================
# ✅ 3) Overpass Query 생성/호출
#    - bbox 순서: (south, west, north, east)
#    - out center qt: way/relation도 중심점 제공 + 빠르게
# =========================================================
def build_overpass_query(south, west, north, east, tags):
    parts = []
    for k, v in tags:
        parts.append(f'node["{k}"="{v}"]({south},{west},{north},{east});')
        parts.append(f'way["{k}"="{v}"]({south},{west},{north},{east});')
        parts.append(f'relation["{k}"="{v}"]({south},{west},{north},{east});')

    q = (
        "[out:json][timeout:180];"
        "(" + "".join(parts) + ");"
        "out center qt;"
    )
    return q

def overpass_fetch_points(north, south, east, west, tags, sleep_sec=SLEEP_SEC, max_retry_per_url=2):
    """
    반환: (lats, lons) numpy arrays
    """
    query = build_overpass_query(south, west, north, east, tags)
    last_err = None

    for url in OVERPASS_URLS:
        for attempt in range(max_retry_per_url):
            try:
                r = requests.post(url, data={"data": query}, timeout=240)

                if r.status_code != 200:
                    last_err = RuntimeError(f"{url} HTTP {r.status_code}: {r.text[:200]}")
                    time.sleep(sleep_sec * (attempt + 1))
                    continue

                js = r.json()
                elems = js.get("elements", [])
                pts = []

                for e in elems:
                    # node
                    if "lat" in e and "lon" in e:
                        pts.append((e["lat"], e["lon"]))
                    # way/relation center
                    elif "center" in e and "lat" in e["center"] and "lon" in e["center"]:
                        pts.append((e["center"]["lat"], e["center"]["lon"]))

                if not pts:
                    time.sleep(sleep_sec)
                    return np.array([]), np.array([])

                pts_arr = np.array(pts, dtype=float)
                pts_arr = np.round(pts_arr, 6)  # 좌표 중복 제거용
                pts_arr = np.unique(pts_arr, axis=0)

                lats = pts_arr[:, 0]
                lons = pts_arr[:, 1]

                time.sleep(sleep_sec)
                return lats, lons

            except Exception as e:
                last_err = e
                time.sleep(sleep_sec * (attempt + 1))
                continue

    print(f"  ⚠️ Overpass 전체 실패: {last_err}")
    return np.array([]), np.array([])

# =========================================================
# ✅ 4) haversine 최소거리(미터) (numpy만)
# =========================================================
EARTH_R = 6371000.0

def haversine_min_m(lat, lon, lats, lons, cand_km=CAND_KM):
    if len(lats) == 0:
        return np.nan

    # 후보 필터(대충 박스)로 속도 개선
    lat_buf = cand_km / 111.0
    lon_buf = cand_km / (111.0 * max(0.2, math.cos(math.radians(lat))))
    mask = (np.abs(lats - lat) <= lat_buf) & (np.abs(lons - lon) <= lon_buf)
    if not mask.any():
        return np.nan

    lats2 = lats[mask]
    lons2 = lons[mask]

    lat1 = np.deg2rad(lat)
    lon1 = np.deg2rad(lon)
    lat2 = np.deg2rad(lats2)
    lon2 = np.deg2rad(lons2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2*np.arcsin(np.sqrt(a))
    return float(EARTH_R * np.min(c))

# =========================================================
# ✅ 5) 도시별 해변 포인트 1번만 수집
#    - 서울은 "해변" 개념이 안 맞으니 아예 스킵(시간 절약)
# =========================================================
city_beaches = {}
cities = df["도시"].dropna().unique().tolist()

for city in cities:
    if city == "서울":
        city_beaches[city] = (np.array([]), np.array([]))
        print("[서울] 해변 수집 스킵 (서울은 해변_10km=0 강제)")
        continue

    sub = df[(df["도시"] == city) & df["위도"].notna() & df["경도"].notna()]
    if len(sub) == 0:
        city_beaches[city] = (np.array([]), np.array([]))
        continue

    lat_min, lat_max = sub["위도"].min(), sub["위도"].max()
    lon_min, lon_max = sub["경도"].min(), sub["경도"].max()
    lat_mean = sub["위도"].mean()

    buf_km = (거리기준_m/1000.0) + EXTRA_KM
    lat_buf, lon_buf = km_to_deg_buffer(lat_mean, km=buf_km)

    north = lat_max + lat_buf
    south = lat_min - lat_buf
    east  = lon_max + lon_buf
    west  = lon_min - lon_buf

    print(f"[{city}] 해변 수집... bbox=({south:.4f},{west:.4f})~({north:.4f},{east:.4f})")

    lats, lons = overpass_fetch_points(north, south, east, west, BEACH_TAGS, sleep_sec=SLEEP_SEC)
    city_beaches[city] = (lats, lons)

    print(f"  ✅ [{city}] 해변 포인트 수: {len(lats)}")

# =========================================================
# ✅ 6) 해변_10km (1/0) 생성
# =========================================================
flag10 = np.zeros(len(df), dtype=int)

for i, row in df.iterrows():
    city = row["도시"]
    lat  = row["위도"]
    lon  = row["경도"]

    # 서울은 무조건 0
    if city == "서울":
        flag10[i] = 0
        continue

    if pd.isna(city) or pd.isna(lat) or pd.isna(lon):
        flag10[i] = 0
        continue

    b_lats, b_lons = city_beaches.get(city, (np.array([]), np.array([])))
    dmin = haversine_min_m(lat, lon, b_lats, b_lons, cand_km=CAND_KM)

    flag10[i] = 1 if (not np.isnan(dmin) and dmin <= 거리기준_m) else 0

df["해변_10km"] = flag10

# =========================================================
# ✅ 7) 최종 컬럼만 저장 (요청대로)
#     - 도시/나라/위도/경도/이용자수/해변_10km
# =========================================================
keep_cols = ["도시", "나라", "위도", "경도"]
if "이용자수" in df.columns:
    keep_cols.append("이용자수")
keep_cols.append("해변_10km")

out_df = df[keep_cols].copy()
out_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("\n저장 완료:", out_path)
print("해변_10km=1:", int((out_df["해변_10km"] == 1).sum()))
print("해변_10km=0:", int((out_df["해변_10km"] == 0).sum()))
print("도시별 해변_10km=1 개수:\n", out_df.groupby("도시")["해변_10km"].sum())


[부산] 해변 수집... bbox=(34.9042,128.6664)~(35.4520,129.4265)
  ✅ [부산] 해변 포인트 수: 147
[서울] 해변 수집 스킵 (서울은 해변_10km=0 강제)
[제주] 해변 수집... bbox=(33.0983,126.0768)~(33.6900,126.9272)
  ✅ [제주] 해변 포인트 수: 576

저장 완료: C:\ai\travel_dataset\전국관광지정보표준데이터_부산서울제주_도시나라위경도_이용자수_해변10km.csv
해변_10km=1: 91
해변_10km=0: 64
도시별 해변_10km=1 개수:
 도시
부산    78
서울     0
제주    13
Name: 해변_10km, dtype: int32


In [ ]:
import pandas as pd
import re

# =========================
# 0) 경로만 수정
# =========================
in_path  = r"C:\ai\travel_dataset\나라도시\대한민국.csv"          # 예: r"C:\ai\travel_dataset\대한민국.csv"
out_path = r"C:\ai\travel_dataset\나라도시\대한민국.csv"  # 예: r"C:\ai\travel_dataset\대한민국_컬럼정리.csv"

# =========================
# 1) CSV 로드(인코딩 자동, 구버전 pandas 호환)
# =========================
def read_csv_auto_encoding(path):
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return pd.read_csv(f)

df = read_csv_auto_encoding(in_path)

# =========================
# 2) 컬럼 탐색 유틸
# =========================
def norm(s):
    return re.sub(r"[^a-z0-9가-힣_]+", "", str(s).strip().lower())

def find_col_by_alias(cols, aliases):
    aliases = set([norm(a) for a in aliases])
    for c in cols:
        if norm(c) in aliases:
            return c
    # 부분 포함도 한 번 시도
    for c in cols:
        nc = norm(c)
        if any(a in nc for a in aliases):
            return c
    return None

cols = list(df.columns)

# 위도/경도 찾기
lat_col = find_col_by_alias(cols, ["위도","lat","latitude","y"])
lon_col = find_col_by_alias(cols, ["경도","lon","lng","longitude","x"])

if lat_col is None or lon_col is None:
    raise ValueError(f"위도/경도 컬럼을 못 찾았어요. 현재 컬럼: {list(df.columns)}")

# 도시/나라 찾기 (없으면 생성)
city_col = find_col_by_alias(cols, ["도시","city","cityname","도시명"])
country_col = find_col_by_alias(cols, ["나라","국가","country"])

# 이용자수 찾기 (없으면 생성)
users_col = find_col_by_alias(cols, ["이용자수","수용인원","capacity","users","usercount","visitor","visitors"])

# 해변 플래그 찾기 (없으면 생성 0)
beach_col = find_col_by_alias(cols, ["해변_10km이내","해변10km이내","해변_10km","해변10km","beach_10km","beach10km"])

# =========================
# 3) 표준 컬럼 만들기
# =========================
out = pd.DataFrame()

out["위도"] = pd.to_numeric(df[lat_col], errors="coerce")
out["경도"] = pd.to_numeric(df[lon_col], errors="coerce")

if users_col is None:
    out["이용자수"] = 0
else:
    out["이용자수"] = pd.to_numeric(df[users_col], errors="coerce").fillna(0).astype(int)

if city_col is None:
    out["도시"] = ""
else:
    out["도시"] = df[city_col].astype(str)

if country_col is None:
    out["나라"] = "대한민국"
else:
    out["나라"] = df[country_col].astype(str)

if beach_col is None:
    out["해변_10km이내"] = 0
else:
    out["해변_10km이내"] = pd.to_numeric(df[beach_col], errors="coerce").fillna(0).astype(int)

# =========================
# 4) 최종 컬럼 순서 고정 + 저장
# =========================
out = out[["위도", "경도", "이용자수", "도시", "나라", "해변_10km이내"]]

out.to_csv(out_path, index=False, encoding="utf-8-sig")
print("저장 완료:", out_path)
print("행 수:", len(out))
print("컬럼:", list(out.columns))


저장 완료: C:\ai\travel_dataset\나라도시\대한민국.csv
행 수: 155
컬럼: ['위도', '경도', '이용자수', '도시', '나라', '해변_10km이내']


In [ ]:
# -*- coding: utf-8 -*-
"""
Korea POI Crawler (CITIES lat/lon) - Category-based, Single CSV, Keep importance_score + osm_url
- Overpass로 도시별 × 카테고리별 POI 수집(각 카테고리 30개)
- ✅ 결과는 "한 파일(통합 CSV)"로만 저장
- ✅ importance_score 컬럼 포함
- ✅ osm_url 컬럼 포함

최종 CSV 컬럼:
country, city, feature, name, lat, lon, importance_score, osm_url

필요: pip install requests pandas
"""

import os
import time
import math
import re
import requests
import pandas as pd


# ============================================================
# 1) 설정
# ============================================================
CITIES = [
    {"city": "서울",  "lat": 37.566535,  "lon": 126.9779692},
    {"city": "부산",  "lat": 35.1795543, "lon": 129.0756416},
    {"city": "제주도","lat": 33.4996213, "lon": 126.5311884,
     "anchors": [
        {"city": "제주시", "lat": 33.4996213, "lon": 126.5311884},
        {"city": "서귀포", "lat": 33.2530,    "lon": 126.5590},
     ]}
]

COUNTRY = "대한민국"
TARGET_PER_CITY_PER_FEATURE = 30

RADIUS_STEPS = [5000, 10000, 20000, 40000, 80000]
ANCHOR_RADIUS_STEPS = [8000, 16000, 32000, 64000, 80000]

SLEEP_SEC = 0.25
MIN_DIST_BETWEEN_PICKED_M = 300

OUT_CSV = r"C:\ai\travel_dataset\나라도시\대한민국.csv"
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

RUN_FEATURES = None

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

SESSION = requests.Session()


# ============================================================
# 2) 필터(숙박/노이즈 최소)
# ============================================================
LODGE_TOURISM = {
    "hotel", "hostel", "guest_house", "motel", "apartment", "resort",
    "chalet", "camp_site", "caravan_site"
}

NAME_BAD_RE = re.compile(
    r"(hotel|khách\s*sạn|resort|hostel|motel|homestay|apartment|villa|inn|"
    r"폐업|입구|놀이터)",
    re.I
)


# ============================================================
# 3) 카테고리(스포츠 없음)
# ============================================================
CATEGORIES = [
    {"feature": "먹거리", "key": "food",
     "blocks": [
         {"kind": "nwr", "tags": ['["amenity"="restaurant"]']},
         {"kind": "nwr", "tags": ['["amenity"="marketplace"]']},
         {"kind": "nwr", "tags": ['["amenity"="food_court"]']},
     ]},

    {"feature": "카페/베이커리", "key": "cafe_bakery",
     "blocks": [
         {"kind": "nwr", "tags": ['["amenity"="cafe"]']},
         {"kind": "nwr", "tags": ['["shop"="bakery"]']},
     ]},

    {"feature": "펍/바", "key": "pub_bar",
     "blocks": [
         {"kind": "nwr", "tags": ['["amenity"="pub"]']},
         {"kind": "nwr", "tags": ['["amenity"="bar"]']},
     ]},

    {"feature": "액티비티", "key": "activity",
     "blocks": [
         # 아웃도어
         {"kind": "relation", "tags": ['["route"="hiking"]']},
         {"kind": "way", "tags": ['["highway"~"^(path|footway|track)$"]']},

         {"kind": "nwr", "tags": ['["sport"="climbing"]']},
         {"kind": "nwr", "tags": ['["climbing"]']},

         {"kind": "nwr", "tags": ['["sport"~"^(surfing|kitesurfing|windsurfing|sailing|rowing|canoe|kayaking|diving|scuba_diving|swimming)$"]']},
         {"kind": "nwr", "tags": ['["leisure"="marina"]']},

         # 레저시설
         {"kind": "nwr", "tags": ['["tourism"="theme_park"]']},
         {"kind": "nwr", "tags": ['["tourism"="zoo"]']},
         {"kind": "nwr", "tags": ['["tourism"="aquarium"]']},
         {"kind": "nwr", "tags": ['["leisure"="bowling_alley"]']},
         {"kind": "nwr", "tags": ['["leisure"="amusement_arcade"]']},
     ]},

    {"feature": "쇼핑", "key": "shopping",
     "blocks": [
         {"kind": "nwr", "tags": ['["shop"="mall"]']},
         {"kind": "nwr", "tags": ['["shop"="department_store"]']},
         {"kind": "nwr", "tags": ['["shop"~"^(gift|craft|handicraft)$"]']},
     ]},

    {"feature": "자연/관광", "key": "nature_sightseeing",
     "blocks": [
         # 랜드마크/역사
         {"kind": "nwr", "tags": ['["tourism"="attraction"]']},
         {"kind": "nwr", "tags": ['["historic"="monument"]']},
         {"kind": "nwr", "tags": ['["historic"~"^(castle|palace|ruins|memorial|fort|archaeological_site)$"]']},

         # 자연 포인트
         {"kind": "nwr", "tags": ['["natural"="water"]["water"="lake"]']},
         {"kind": "nwr", "tags": ['["waterway"="waterfall"]']},
         {"kind": "nwr", "tags": ['["natural"="waterfall"]']},
         {"kind": "nwr", "tags": ['["natural"="peak"]']},
         {"kind": "nwr", "tags": ['["waterway"~"^(river|riverbank)$"]']},

         # 공원(식물원/정원/국립공원)
         {"kind": "nwr", "tags": ['["tourism"="botanical_garden"]']},
         {"kind": "nwr", "tags": ['["leisure"="garden"]']},
         {"kind": "relation", "tags": ['["boundary"="national_park"]']},

         # 웰니스
         {"kind": "nwr", "tags": ['["amenity"="spa"]']},
         {"kind": "nwr", "tags": ['["leisure"="sauna"]']},
         {"kind": "nwr", "tags": ['["natural"="hot_spring"]']},
     ]},
]

KEEP_COLS = ["country", "city", "feature", "name", "lat", "lon", "importance_score", "osm_url"]


# ============================================================
# 4) 유틸
# ============================================================
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlon/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def normalize_name(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s가-힣]", "", s)
    return s

def importance_score(tags: dict) -> int:
    s = 0
    if "wikidata" in tags: s += 50
    if "wikipedia" in tags: s += 40
    if "heritage" in tags: s += 25
    if "website" in tags: s += 5
    return int(s)

def pick_name(tags: dict) -> str:
    return (tags.get("name") or tags.get("name:ko") or tags.get("name:en") or "").strip()


# ============================================================
# 5) Overpass
# ============================================================
def build_overpass_query(lat: float, lon: float, radius_m: int, blocks: list):
    lines = ["[out:json][timeout:60];", "("]
    for b in blocks:
        kind = b.get("kind", "nwr")
        tag_str = "".join(b.get("tags", []))

        need_name = (kind in ["way", "relation"]) or ('["highway"' in tag_str) or ('["route"' in tag_str)
        name_filter = '["name"]' if need_name else ""

        lines.append(f'  {kind}(around:{radius_m},{lat},{lon}){tag_str}{name_filter};')
    lines.append(");")
    lines.append("out center tags;")
    return "\n".join(lines)

def overpass_fetch(query: str, max_retry=3):
    last_err = None
    for endpoint in OVERPASS_URLS:
        for attempt in range(1, max_retry + 1):
            try:
                r = SESSION.post(endpoint, data={"data": query}, timeout=120)
                if r.status_code == 200:
                    return r.json()
                last_err = f"{endpoint} status={r.status_code} body={r.text[:200]}"
                time.sleep(1.0 * attempt)
            except Exception as e:
                last_err = f"{endpoint} error={e}"
                time.sleep(1.0 * attempt)
    raise RuntimeError(f"❌ Overpass 요청 실패: {last_err}")

def parse_elements(data):
    elements = data.get("elements", []) if isinstance(data, dict) else []
    out = []
    for e in elements:
        tags = e.get("tags", {}) or {}

        name = pick_name(tags)
        if not name:
            continue
        if NAME_BAD_RE.search(name):
            continue

        tourism = (tags.get("tourism") or "").lower()
        if tourism in LODGE_TOURISM:
            continue

        lat = e.get("lat"); lon = e.get("lon")
        if lat is None or lon is None:
            center = e.get("center") or {}
            lat, lon = center.get("lat"), center.get("lon")
        if lat is None or lon is None:
            continue

        osm_type = e.get("type")
        osm_id = e.get("id")

        out.append({
            "name": name,
            "lat": float(lat),
            "lon": float(lon),
            "tags": dict(tags),
            "osm_type": osm_type,
            "osm_id": osm_id,
            "osm_url": f"https://www.openstreetmap.org/{osm_type}/{osm_id}",
        })
    return out


# ============================================================
# 6) 선별/중복 제거
# ============================================================
def too_close_to_picked(picked_rows, lat, lon, min_dist_m):
    for p in picked_rows:
        if haversine_m(p["lat"], p["lon"], lat, lon) < min_dist_m:
            return True
    return False

def rank_and_pick(city_name: str, clat: float, clon: float, feature: str, candidates: list, target_n: int):
    for c in candidates:
        c["_distance_m"] = haversine_m(clat, clon, c["lat"], c["lon"])
        c["_importance"] = importance_score(c["tags"])
    candidates.sort(key=lambda x: (-x["_importance"], x["_distance_m"]))

    picked = []
    seen_osm = set()
    seen_name_loc = set()

    for c in candidates:
        if len(picked) >= target_n:
            break

        uniq_osm = (c["osm_type"], c["osm_id"])
        if uniq_osm in seen_osm:
            continue

        key2 = (normalize_name(c["name"]), round(c["lat"], 5), round(c["lon"], 5))
        if key2 in seen_name_loc:
            continue

        if too_close_to_picked(picked, c["lat"], c["lon"], MIN_DIST_BETWEEN_PICKED_M):
            continue

        seen_osm.add(uniq_osm)
        seen_name_loc.add(key2)

        picked.append({
            "country": COUNTRY,
            "city": city_name,
            "feature": feature,
            "name": c["name"],
            "lat": c["lat"],
            "lon": c["lon"],
            "importance_score": int(c["_importance"]),
            "osm_url": c["osm_url"],
        })

    return picked


# ============================================================
# 7) 수집(도시×카테고리)
# ============================================================
def collect_city_feature(city_obj: dict, cat: dict):
    city = city_obj["city"]
    clat = float(city_obj["lat"])
    clon = float(city_obj["lon"])

    feature = cat["feature"]
    blocks = cat["blocks"]

    anchors = city_obj.get("anchors")
    centers = []
    if anchors:
        for a in anchors:
            centers.append((a["city"], float(a["lat"]), float(a["lon"]), ANCHOR_RADIUS_STEPS))
    else:
        centers.append((city, clat, clon, RADIUS_STEPS))

    all_candidates = []
    uniq = set()

    for center_name, lat, lon, steps in centers:
        for radius in steps:
            q = build_overpass_query(lat, lon, radius, blocks)
            data = overpass_fetch(q, max_retry=2)
            cand = parse_elements(data)

            added = 0
            for c in cand:
                k = (c["osm_type"], c["osm_id"])
                if k in uniq:
                    continue
                uniq.add(k)
                all_candidates.append(c)
                added += 1

            print(f"  - {city}/{feature} anchor={center_name} radius={radius}m +{added} (누적 {len(all_candidates)})")
            time.sleep(SLEEP_SEC)

            if len(all_candidates) >= max(80, TARGET_PER_CITY_PER_FEATURE * 3):
                break

    return rank_and_pick(city, clat, clon, feature, all_candidates, TARGET_PER_CITY_PER_FEATURE)


# ============================================================
# 8) 실행 + 통합 CSV 저장
# ============================================================
def main():
    all_rows = []

    for city_obj in CITIES:
        print("\n" + "=" * 70)
        print(f"▶ 도시 시작: {city_obj['city']}")

        for cat in CATEGORIES:
            feature = cat["feature"]
            if RUN_FEATURES is not None and feature not in RUN_FEATURES:
                continue

            print(f"\n▶ {city_obj['city']} / {feature} 수집 시작 (목표 {TARGET_PER_CITY_PER_FEATURE})")
            try:
                rows = collect_city_feature(city_obj, cat)
                print(f"✅ {city_obj['city']} / {feature} 완료: {len(rows)}건")
                all_rows.extend(rows)

                # ✅ 중간 저장
                df_tmp = pd.DataFrame(all_rows)
                df_tmp = df_tmp[KEEP_COLS]
                df_tmp.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
                print(f"📁 중간 저장: {OUT_CSV} (누적 {len(df_tmp)}건)")
            except Exception as e:
                print(f"❌ {city_obj['city']} / {feature} 실패(스킵): {e}")
                continue

    if not all_rows:
        print("\n❌ 최종 0건입니다. (Overpass 장애/네트워크/차단 가능)")
        return

    df = pd.DataFrame(all_rows)
    df = df[KEEP_COLS].copy()
    df = df.sort_values(["city", "feature", "importance_score"], ascending=[True, True, False])

    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ 최종 CSV 저장 완료: {OUT_CSV}")
    print(df.groupby(["city", "feature"])["name"].count())

if __name__ == "__main__":
    main()
